# SSSL-Based Continual IDS on AnoShift (NeurIPS 2022 Benchmark)
### VS Code + Google Colab Remote Kernel Guide

This notebook trains and evaluates the **Self-Supervised Semi-Supervised Learning (SSSL)** Intrusion Detection System with **Gradient Projection Memory (GPM)** on the **AnoShift** longitudinal distribution shift benchmark.

### Supported Continual Task Heads:
1. **Task 1: General Intrusion Head** (`--task intrusion`) &mdash; Malicious flow detection vs benign traffic.
2. **Task 2: Denial of Service (DoS) Head** (`--task dos`) &mdash; SYN floods, connection floods, and error bursts.
3. **Task 3: Port Scan Head** (`--task port_scan`) &mdash; Reconnaissance, host sweeps, and probing.
4. **Zero-Day Honeypot Detector** (`evaluate_zeroday.py`) &mdash; Autoencoder catches unknown honeypot behavioral anomalies (Kyoto label `-2`).

## Step 1: Environment Setup & Sync Repository on Colab VM

In [27]:
import os, sys
import tensorflow as tf

print("Python Executable:", sys.executable)
print("TensorFlow Version:", tf.__version__)
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

# Automatic Colab VM Workspace Sync
if 'google.colab' in sys.modules or os.path.exists('/content'):
    repo_dir = '/content/sssl-continual-ids'
    if not os.path.exists(repo_dir):
        print("\n[INFO] Cloning sssl-continual-ids repository into Colab VM...")
        !git clone -b Experimental_Analysis https://github.com/yaswanthvuppala/sssl-continual-ids.git /content/sssl-continual-ids
    else:
        print("\n[INFO] Syncing latest updates from GitHub...")
        !cd /content/sssl-continual-ids && git fetch origin Experimental_Analysis && git reset --hard origin/Experimental_Analysis
    
    %cd /content/sssl-continual-ids/ids-system
else:
    # Local VS Code environment fallback
    if os.path.exists('ids-system'):
        %cd ids-system
    elif os.path.basename(os.getcwd()) != 'ids-system' and os.path.exists('../ids-system'):
        %cd ../ids-system

print("\nActive Working Directory:", os.getcwd())
!git log -1 --oneline

Python Executable: /usr/bin/python3
TensorFlow Version: 2.20.0
GPU Devices Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[INFO] Syncing latest updates from GitHub...
From https://github.com/yaswanthvuppala/sssl-continual-ids
 * branch            Experimental_Analysis -> FETCH_HEAD
HEAD is now at f15c9ef Fix load_dataset arguments in train_ssl and train_task
/content/sssl-continual-ids/ids-system

Active Working Directory: /content/sssl-continual-ids/ids-system
f15c9ef (HEAD -> Experimental_Analysis, origin/Experimental_Analysis) Fix load_dataset arguments in train_ssl and train_task


## Step 2: Install Parquet & Project Dependencies

In [38]:
!pip install -q pyarrow fastparquet scikit-learn seaborn matplotlib tqdm pyyaml

## Step 3: Download AnoShift Benchmark Dataset

In [ ]:
# Download official AnoShift Subset I/10 (~150MB, fast and recommended)
!python data/download_anoshift.py --subset I/10 --save_dir ./data/anoshift

# Note: To download the larger Subset I/33 (~500MB) instead, run:
# !python data/download_anoshift.py --subset I/33 --save_dir ./data/anoshift

## Step 4: Self-Supervised Learning (SSL) Pretraining on In-Distribution Normal Traffic

In [41]:
!cd /content/sssl-continual-ids && git fetch origin Experimental_Analysis && git reset --hard origin/Experimental_Analysis

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 801 bytes | 400.00 KiB/s, done.
From https://github.com/yaswanthvuppala/sssl-continual-ids
 * branch            Experimental_Analysis -> FETCH_HEAD
   4c8342d..3da466b  Experimental_Analysis -> origin/Experimental_Analysis
HEAD is now at 3da466b Fix train_ssl.py and train_task.py: add --max_samples arg and load_dataset arguments


In [42]:
!python training/train_ssl.py \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --label_col Label \
    --epochs 15 \
    --batch_size 256 \
    --dataset_name anoshift

Loading 20 AnoShift train data file(s)...
  [1/20] Loading 2006_full.parquet... (5,000 rows)
  [2/20] Loading 2006_full_valid.parquet... (5,000 rows)
  [3/20] Loading 2007_full.parquet... (5,000 rows)
  [4/20] Loading 2007_full_valid.parquet... (5,000 rows)
  [5/20] Loading 2008_full.parquet... (5,000 rows)
  [6/20] Loading 2008_full_valid.parquet... (5,000 rows)
  [7/20] Loading 2009_full.parquet...^C


## Step 5: Continual Learning for ALL 3 Task Heads (with GPM Protection)

In [ ]:
# 5.1 Train Task 1: General Intrusion Head
print("\n=== [1/3] Training Task: Intrusion ===")
!python training/train_task.py \
    --task intrusion \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 20 \
    --unfreeze_encoder \
    --encoder_lr 0.003 \
    --dataset_name anoshift

# 5.2 Train Task 2: DoS Head (protecting Intrusion representations via GPM)
print("\n=== [2/3] Training Task: DoS (with GPM) ===")
!python training/train_task.py \
    --task dos \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 15 \
    --unfreeze_encoder \
    --encoder_lr 0.003 \
    --dataset_name anoshift

# 5.3 Train Task 3: Port Scan Head (protecting Intrusion + DoS representations via GPM)
print("\n=== [3/3] Training Task: Port Scan (with GPM) ===")
!python training/train_task.py \
    --task port_scan \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 15 \
    --unfreeze_encoder \
    --encoder_lr 0.003 \
    --dataset_name anoshift

## Step 6: Train Anomaly Autoencoder for Unknown / Zero-Day Honeypot Threats

In [ ]:
!python training/train_anomaly.py \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --epochs 20 \
    --latent_dim 16 \
    --dataset_name anoshift

## Step 7: Temporal Distribution Shift Benchmark Evaluation (IN vs NEAR vs FAR)

In [ ]:
# 7.1 Evaluate All 3 Heads on In-Distribution (2006-2010)
print("\n=== Evaluating In-Distribution (IN) Split for ALL 3 Heads ===")
!python training/evaluate.py --task all --dataset anoshift --data_path ./data/anoshift --split iid --dataset_name anoshift

# 7.2 Evaluate Near-Distribution (2011-2013 Moderate Shift)
print("\n=== Evaluating Near-Distribution (NEAR) Split ===")
!python training/evaluate.py --task intrusion --dataset anoshift --data_path ./data/anoshift --split near --dataset_name anoshift

# 7.3 Evaluate Far-Distribution (2014-2015 Severe Shift)
print("\n=== Evaluating Far-Distribution (FAR) Split ===")
!python training/evaluate.py --task intrusion --dataset anoshift --data_path ./data/anoshift --split far --dataset_name anoshift

## Step 8: Evaluate Zero-Day Honeypot Detection

In [ ]:
# Tests unknown honeypot behavioral anomalies (Kyoto label -2) against known attacks
!python training/evaluate_zeroday.py \
    --dataset anoshift \
    --data_path ./data/anoshift \
    --dataset_name anoshift

## Step 9: Generate Publication Visualizations & Backward Transfer Matrix

In [ ]:
# Generate visualization dashboards for all 3 tasks
!python training/visualize_metrics.py --task intrusion --dataset_name anoshift
!python training/visualize_metrics.py --task dos --dataset_name anoshift
!python training/visualize_metrics.py --task port_scan --dataset_name anoshift

# Compute BWT & FWT Transfer Matrix
!python training/compute_transfer.py --dataset_name anoshift --data_path ./data/anoshift

# Display plot
from IPython.display import Image, display
for p in ["dashboard_intrusion.png", "dashboard_dos.png", "dashboard_port_scan.png"]:
    path = f"logs/anoshift/plots/{p}"
    if os.path.exists(path):
        print(f"\nDisplaying {p}:")
        display(Image(filename=path))